# T02 — 운동학: 좌표계와 부호 규약, 분절각·관절각, 2D에서 3D로

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait-theory/note-T02.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait-theory/note-T02.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 본문

### 7. 각도의 미분은 각속도가 아니다

아래 코드는 이 노트를 위해 새로 작성했습니다. 가상의 무릎 자세로 ① 각 되찾기, ② 순서 의존성, ③ 각도 변화율 vs 각속도, ④ gimbal lock을 확인합니다.

In [ ]:
import numpy as np

def Rx(a): c, s = np.cos(a), np.sin(a); return np.array([[1, 0, 0], [0, c, s], [0, -s, c]])
def Ry(b): c, s = np.cos(b), np.sin(b); return np.array([[c, 0, -s], [0, 1, 0], [s, 0, c]])
def Rz(g): c, s = np.cos(g), np.sin(g); return np.array([[c, s, 0], [-s, c, 0], [0, 0, 1]])

def cardan_xyz(a, b, g):          # 근위 → 원위 변환, x(굴곡) → y(외전) → z(축회전) 순서
    return Rz(g) @ Ry(b) @ Rx(a)

def angles_xyz(R):                # 행렬에서 세 각을 되찾기
    a = np.arctan2(-R[2, 1], R[2, 2])
    b = np.arcsin(R[2, 0])
    g = np.arctan2(-R[1, 0], R[0, 0])
    return np.degrees([a, b, g])

ang = np.radians([40.0, 10.0, -8.0])   # 굴곡 40°, 외전 10°, 축회전 -8° (가상의 무릎 자세)
R = cardan_xyz(*ang)
print("되찾은 각(°):", np.round(angles_xyz(R), 3))

# 1) 순서를 바꾸면 같은 세 숫자가 다른 자세가 된다
R_zyx = Rx(ang[0]) @ Ry(ang[1]) @ Rz(ang[2])
print("XYZ와 ZYX 행렬 원소 차이의 최댓값:", round(np.abs(R - R_zyx).max(), 3))

# 2) 각도의 시간미분 ≠ 각속도
t = np.linspace(0, 1, 2001); dt = t[1] - t[0]
a_t = np.radians(40 + 20*np.sin(2*np.pi*t))
b_t = np.radians(10 + 8*np.sin(2*np.pi*t + 1))
g_t = np.radians(-8 + 6*np.sin(2*np.pi*t + 2))
i = 700
Rm, Rp = cardan_xyz(a_t[i-1], b_t[i-1], g_t[i-1]), cardan_xyz(a_t[i+1], b_t[i+1], g_t[i+1])
Rdot = (Rp - Rm) / (2*dt)
Ri = cardan_xyz(a_t[i], b_t[i], g_t[i])
W = Ri @ Rdot.T                    # 원위 분절 축에서 본 각속도의 반대칭 행렬
w_true = np.array([W[2, 1], W[0, 2], W[1, 0]])
ad, bd, gd = [np.gradient(x, dt)[i] for x in (a_t, b_t, g_t)]
c2, s2 = np.cos(b_t[i]), np.sin(b_t[i]); c3, s3 = np.cos(g_t[i]), np.sin(g_t[i])
w_formula = np.array([c2*c3*ad + s3*bd, -c2*s3*ad + c3*bd, s2*ad + gd])
print("각도 변화율 (rad/s):     ", np.round([ad, bd, gd], 3))
print("각속도(회전행렬 차분):   ", np.round(w_true, 3))
print("각속도(변환식):          ", np.round(w_formula, 3))

# 3) gimbal lock: 둘째 각이 90°이면 첫째·셋째 각을 구분할 수 없다
for a, g in [(30, 0), (0, 30), (20, 10)]:
    Rg = cardan_xyz(np.radians(a), np.radians(90), np.radians(g))
    print(f"a={a:>3}°, g={g:>4}° → 행렬 첫 줄", np.round(Rg[0], 3), " 셋째 줄", np.round(Rg[2], 3))